# 09 — Live feed reconciliation (#154)

> **Record only.** The PlayByPlayV3 archive, its fetcher and `PLAY_BY_PLAY_SCHEMA` retired once recaps moved onto the feed; the reports under `reference/live_feed_reconciliation/` are the comparison's record and this notebook no longer runs.

**Objective:** decide whether the liveData play-by-play (cdn.nba.com) can replace
the PlayByPlayV3 archive (stats.nba.com) under recaps, by comparing the two over
every game of both seasons.

**Deliverable:** per season, under `data/<season>/reference/live_feed_reconciliation/`
(gitignored, reference only, never published):

- `report.md`: every table below, with the reading
- `exceptions.csv`: every archive row that is not plainly matched and every
  unexpected feed row, one per line

**The replacement stands when:**

1. every listed game is banked in the feed archive
2. no archive row is left unexplained
3. no unexpected feed row is left unexplained
4. every feed row carries a timestamp and a score

**Inputs:** `reference/play_by_play/` (scripts.fetch_play_by_play),
`reference/play_by_play_live/` (scripts.fetch_live_play_by_play), and the banked
game logs. This notebook reads both seasons by path, whichever season is active.

In [1]:
import os
import sys
from datetime import date
from pathlib import Path

# Bootstrap: run from anywhere (walk up to the repo root, chdir).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
import polars.selectors as cs

from pipeline.games import (
    PLAYER_GAME_LOG_FILENAME,
    TEAM_GAME_LOG_FILENAME,
    build_games,
)
from pipeline.schemas import LIVE_PLAY_BY_PLAY_SCHEMA, PLAY_BY_PLAY_SCHEMA
from utils.constants import (
    LIVE_PLAY_BY_PLAY_SUBDIR,
    PLAY_BY_PLAY_SUBDIR,
    REFERENCE_DATA_SUBDIR,
)
from utils.paths import get_data_dir
from utils.team_config import load_team_config

SEASONS = ["2025-26", "2024-25"]  # both archives, every game
REPORT_SUBDIR = "live_feed_reconciliation"
SPOT_CHECK_GAME = "0042500317"  # 2025-26 WCF G7

INSTANT_REPLAY = "Instant Replay"
GAME_TZ = "America/New_York"
REGULATION_PERIODS, PERIOD_SECONDS, OVERTIME_SECONDS = 4, 720, 300
CLOCK = r"^PT(\d+)M(\d+(?:\.\d+)?)S$"
WALL_CLOCK = r"\((\d{1,2}):(\d{2}) (AM|PM) E[SD]T\)"

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(240)
pl.Config.set_tbl_hide_dataframe_shape(True)


def reference_dir(season: str) -> Path:
    """A season's reference directory, whichever season is active."""
    return get_data_dir(season) / REFERENCE_DATA_SUBDIR


def clock_seconds(col: str) -> pl.Expr:
    """Seconds left in the period, from the ISO clock both archives carry."""
    parts = pl.col(col).str.extract_groups(CLOCK)
    return parts.struct.field("1").cast(pl.Float64) * 60 + parts.struct.field(
        "2"
    ).cast(pl.Float64)


def spread(frame: pl.DataFrame, col: str) -> pl.DataFrame:
    """The distribution of one numeric column, one row."""
    c = pl.col(col)
    return frame.select(
        pl.len().alias("n"),
        c.min().round(1).alias("min"),
        c.quantile(0.05).round(1).alias("p05"),
        c.median().round(1).alias("median"),
        c.mean().round(1).alias("mean"),
        c.quantile(0.95).round(1).alias("p95"),
        c.max().round(1).alias("max"),
    )

## 1. Load and coverage

Both archives are cut to the games banked in both. `kind` puts the two
vocabularies on one: the archive's `Made Shot` and the feed's `2pt` are both a
`shot`. `actor` is the player, or the team on a team row.

In [2]:
ARCHIVE_COLUMNS = [
    "game_id", "action_number", "action_id", "period", "clock", "team_id",
    "person_id", "action_type", "sub_type", "description",
]
FEED_COLUMNS = [
    "game_id", "action_number", "order_number", "period", "clock", "time_actual",
    "team_id", "person_id", "action_type", "sub_type", "description",
    "score_home", "score_away", "qualifiers", "block_person_id", "steal_person_id",
    "player_name_i", "descriptor", "points_total", "rebound_total",
    "assist_person_id", "assist_total", "turnover_total", "foul_personal_total",
]

# What kind of action a row is, on one vocabulary. The archive leaves
# action_type blank on a block or a steal and names it in the description.
ARCHIVE_KIND = (
    pl.when(pl.col("action_type").is_in(["Made Shot", "Missed Shot"]))
    .then(pl.lit("shot"))
    .when((pl.col("action_type") == "") & pl.col("description").str.contains("BLOCK"))
    .then(pl.lit("block"))
    .when((pl.col("action_type") == "") & pl.col("description").str.contains("STEAL"))
    .then(pl.lit("steal"))
    .otherwise(pl.col("action_type").str.to_lowercase().str.replace_all(" ", ""))
)
# The feed splits a substitution into an out row and an in row; the
# archive's one row carries the player going out, so the out row is its
# counterpart and the in row is a kind of its own.
FEED_KIND = (
    pl.when(pl.col("action_type").is_in(["2pt", "3pt"]))
    .then(pl.lit("shot"))
    .when((pl.col("action_type") == "substitution") & (pl.col("sub_type") == "in"))
    .then(pl.lit("substitution in"))
    .otherwise(pl.col("action_type"))
)
# Who acted: the player, or the team on a team row. The archive puts the
# team id in person_id on most team rows and in team_id on a heave; the
# feed leaves person_id 0 and fills team_id.
ACTOR = (
    pl.when(pl.col("person_id") != 0)
    .then(pl.col("person_id"))
    .otherwise(pl.col("team_id").fill_null(0))
)


def load_season(season: str) -> dict:
    """Both archives for one season, cut to the games banked in both."""
    ref = reference_dir(season)
    abbr_to_team = {i["abbreviation"]: t for t, i in load_team_config().items()}
    listed = build_games(
        pl.read_parquet(ref / TEAM_GAME_LOG_FILENAME), abbr_to_team
    )["game_id"].to_list()

    archive_ids = {p.stem for p in (ref / PLAY_BY_PLAY_SUBDIR).glob("*.parquet")}
    feed_ids = {p.stem for p in (ref / LIVE_PLAY_BY_PLAY_SUBDIR).glob("*.parquet")}
    both = sorted(set(listed) & archive_ids & feed_ids)

    archive = (
        pl.scan_parquet(
            [ref / PLAY_BY_PLAY_SUBDIR / f"{g}.parquet" for g in both],
            schema=PLAY_BY_PLAY_SCHEMA,
        )
        .select(ARCHIVE_COLUMNS)
        .with_columns(
            ARCHIVE_KIND.alias("kind"),
            ACTOR.alias("actor"),
            clock_seconds("clock").alias("clock_s"),
        )
        .collect()
    )
    feed = (
        pl.scan_parquet(
            [ref / LIVE_PLAY_BY_PLAY_SUBDIR / f"{g}.parquet" for g in both],
            schema=LIVE_PLAY_BY_PLAY_SCHEMA,
        )
        .select(FEED_COLUMNS)
        .with_columns(
            FEED_KIND.alias("kind"),
            ACTOR.alias("actor"),
            clock_seconds("clock").alias("clock_s"),
            pl.col("time_actual")
            .str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ", time_zone="UTC", strict=False)
            .alias("stamped"),
        )
        .collect()
        .sort("game_id", "order_number")
    )
    box = pl.read_parquet(ref / PLAYER_GAME_LOG_FILENAME).filter(
        pl.col("game_id").is_in(both)
    )
    return {
        "season": season,
        "box": box,
        "listed": listed,
        "archive_only": sorted((set(listed) & archive_ids) - feed_ids),
        "feed_only": sorted((set(listed) & feed_ids) - archive_ids),
        "neither": sorted(set(listed) - archive_ids - feed_ids),
        "both": both,
        "archive": archive,
        "feed": feed,
    }


DATA = {season: load_season(season) for season in SEASONS}
coverage = pl.DataFrame(
    [
        {
            "season": s,
            "games listed": len(d["listed"]),
            "in both archives": len(d["both"]),
            "archive only": len(d["archive_only"]),
            "feed only": len(d["feed_only"]),
            "in neither": len(d["neither"]),
            "archive rows": d["archive"].height,
            "feed rows": d["feed"].height,
        }
        for s, d in DATA.items()
    ]
)
for d in DATA.values():
    repeated = d["feed"].group_by("game_id", "action_number").len().filter(pl.col("len") > 1)
    assert repeated.height == 0, f"{d['season']}: feed action_number repeats: {repeated.head()}"
coverage

season,games listed,in both archives,archive only,feed only,in neither,archive rows,feed rows
str,i64,i64,i64,i64,i64,i64,i64
"""2025-26""",1387,1387,0,0,0,703459,801263
"""2024-25""",1390,1390,0,0,0,687151,777231


## 2. Archive rows against the feed

Every archive row gets one status. The first pass looks at the feed row on the
same action number; the later passes look for a row the number did not settle
among the feed rows no archive row has claimed.

| status | meaning |
|---|---|
| matched | the number finds the same kind of action, same player, period and clock |
| clock differs | same action, the clock is off |
| period differs | same action, another period |
| under another number | same period, clock, player and kind, on another number |
| under another number, other clock | same period, player and kind; nearest on the clock |
| under another number, other player | same period, clock and kind; another player credited |
| player differs | the number finds the same kind of action, credited to another player |
| kind differs | the number finds another kind of action |
| not found | none of the above |
| instant replay | set aside; the feed does not carry them |

A block or a steal in the archive has a blank `action_type` and shares its number
with the shot or turnover it ended. It matches when that feed row names the same
defender in `block_person_id` or `steal_person_id`.

In [3]:
KEY = ["game_id", "action_number"]
# The later passes, in order: what an open archive row and an unclaimed
# feed row must share to be the same action under another number.
PASSES = {
    "under another number": ["game_id", "period", "kind", "actor", "clock_s"],
    "under another number, other clock": ["game_id", "period", "kind", "actor"],
    "under another number, other player": ["game_id", "period", "kind", "clock_s"],
}
OTHER_CLOCK = "under another number, other clock"
SETTLED = ["matched", "clock differs", "period differs"]  # the number finds the same action
STATUS_ORDER = [
    "matched",
    "clock differs",
    "period differs",
    *PASSES,
    "player differs",
    "kind differs",
    "not found",
    "instant replay",
]


def on_the_number(archive: pl.DataFrame, feed: pl.DataFrame) -> pl.DataFrame:
    """First pass: the feed row that carries the archive row's action number."""
    joined = archive.join(
        feed.select(
            *KEY,
            pl.col("period").alias("feed_period"),
            pl.col("clock").alias("feed_clock"),
            pl.col("clock_s").alias("feed_clock_s"),
            pl.col("kind").alias("feed_kind"),
            pl.col("actor").alias("feed_actor"),
            pl.col("description").alias("feed_description"),
            "block_person_id",
            "steal_person_id",
        ),
        on=KEY,
        how="left",
    )
    # A block or steal rides on the number of the play it ended, which
    # names the defender in a field. The archive leaves the defender 0 on
    # a blocked heave.
    defender = (
        pl.when(pl.col("kind") == "block")
        .then(pl.col("block_person_id"))
        .otherwise(pl.col("steal_person_id"))
    )
    is_defense = pl.col("kind").is_in(["block", "steal"])
    same_kind = (
        pl.when(is_defense)
        .then(defender.is_not_null())
        .otherwise(pl.col("kind") == pl.col("feed_kind"))
    )
    same_player = (
        pl.when(is_defense)
        .then((defender == pl.col("actor")) | (pl.col("actor") == 0))
        # On a jump ball the archive names one jumper, the feed the player
        # the tip went to. On a heave the feed names the team only.
        .when(pl.col("kind").is_in(["jumpball", "heave"]))
        .then(True)
        .otherwise(pl.col("actor") == pl.col("feed_actor"))
    )
    return joined.with_columns(
        pl.when(pl.col("action_type") == INSTANT_REPLAY)
        .then(pl.lit("instant replay"))
        .when(pl.col("feed_kind").is_null())
        .then(pl.lit("not found"))
        .when(~same_kind)
        .then(pl.lit("kind differs"))
        .when(~same_player)
        .then(pl.lit("player differs"))
        .when(pl.col("period") != pl.col("feed_period"))
        .then(pl.lit("period differs"))
        .when(pl.col("clock_s") != pl.col("feed_clock_s"))
        .then(pl.lit("clock differs"))
        .otherwise(pl.lit("matched"))
        .alias("status"),
        pl.col("action_number").alias("feed_action_number"),
    )


def feed_side(free: pl.DataFrame) -> pl.DataFrame:
    """Unclaimed feed rows under the names a pairing carries them by."""
    return free.select(
        "game_id",
        "period",
        "kind",
        "actor",
        "clock_s",
        "order_number",
        pl.col("action_number").alias("other_number"),
        pl.col("clock").alias("other_clock"),
        pl.col("clock_s").alias("other_clock_s"),
        pl.col("kind").alias("other_kind"),
        pl.col("actor").alias("other_actor"),
        pl.col("description").alias("other_description"),
    )


def elsewhere(open_rows: pl.DataFrame, free: pl.DataFrame, where: list[str]) -> pl.DataFrame:
    """Pair open archive rows with unclaimed feed rows that share `where`.

    Rows that share it pair in order, so two free throws at one clock line up.
    """
    nth = pl.int_range(pl.len()).over(where)
    candidates = (
        feed_side(free)
        .sort("order_number")
        .with_columns(nth.alias("nth"))
        .select(*where, "nth", cs.starts_with("other_"))
    )
    return (
        open_rows.sort("action_id")
        .with_columns(nth.alias("nth"))
        .join(candidates, on=[*where, "nth"], how="inner")
        .select("game_id", "action_id", cs.starts_with("other_"))
    )


def nearest(open_rows: pl.DataFrame, free: pl.DataFrame, where: list[str]) -> pl.DataFrame:
    """Pair each open archive row with the unclaimed feed row nearest on the clock.

    A feed row pairs once: with the archive row closest to it.
    """
    candidates = feed_side(free).drop("order_number").sort("clock_s")
    gap = (pl.col("clock_s") - pl.col("other_clock_s")).abs()
    return (
        open_rows.sort("clock_s")
        .join_asof(
            candidates, on="clock_s", by=where, strategy="nearest", check_sortedness=False
        )
        .filter(pl.col("other_number").is_not_null())
        .sort(gap)
        .unique(["game_id", "other_number"], keep="first", maintain_order=True)
        .select("game_id", "action_id", cs.starts_with("other_"))
    )


def match_archive_rows(archive: pl.DataFrame, feed: pl.DataFrame) -> pl.DataFrame:
    """Give every archive row one status against the feed."""
    rows = on_the_number(archive, feed)
    row_key = ["game_id", "action_id"]

    # Feed rows no archive row has claimed: a settled row claims the row on
    # its number, except a block or steal, which borrowed the number.
    claimed = rows.filter(
        pl.col("status").is_in(SETTLED) & ~pl.col("kind").is_in(["block", "steal"])
    )
    free = feed.join(claimed.select(KEY), on=KEY, how="anti")
    open_rows = rows.filter(pl.col("status").is_in(["not found", "kind differs"]))

    found = []
    for status, where in PASSES.items():
        pair = nearest if status == OTHER_CLOCK else elsewhere
        pairs = pair(open_rows, free, where).with_columns(pl.lit(status).alias("found"))
        found.append(pairs)
        open_rows = open_rows.join(pairs.select(row_key), on=row_key, how="anti")
        free = free.join(
            pairs.select("game_id", pl.col("other_number").alias("action_number")),
            on=KEY,
            how="anti",
        )

    moved = pl.col("found").is_not_null()
    swap = {
        "feed_action_number": "other_number",
        "feed_clock": "other_clock",
        "feed_clock_s": "other_clock_s",
        "feed_kind": "other_kind",
        "feed_actor": "other_actor",
        "feed_description": "other_description",
    }
    return (
        rows.join(pl.concat(found), on=row_key, how="left")
        .with_columns(
            pl.coalesce("found", "status").alias("status"),
            *[
                pl.when(moved).then(pl.col(other)).otherwise(pl.col(col)).alias(col)
                for col, other in swap.items()
            ],
        )
        .with_columns((pl.col("clock_s") - pl.col("feed_clock_s")).alias("clock_gap_s"))
        .drop("found", cs.starts_with("other_"))
    )


def by_season(table) -> pl.DataFrame:
    """One long frame from a per-season table builder, season first."""
    return pl.concat(
        [table(d).with_columns(pl.lit(s).alias("season")) for s, d in DATA.items()],
        how="diagonal_relaxed",
    )


def across_seasons(counts: pl.DataFrame, index: str) -> pl.DataFrame:
    """A count per season, side by side; a season without the row reads 0."""
    wide = counts.pivot(on="season", index=index, values="len").fill_null(0)
    absent = [pl.lit(0, dtype=pl.UInt32).alias(s) for s in SEASONS if s not in wide.columns]
    return wide.with_columns(absent).select(index, *SEASONS)


for d in DATA.values():
    d["rows"] = match_archive_rows(d["archive"], d["feed"])
    assert d["rows"].height == d["archive"].height, "a row was dropped or doubled"

status_counts = (
    across_seasons(by_season(lambda d: d["rows"].group_by("status").len()), "status")
    .with_columns(pl.col("status").cast(pl.Enum(STATUS_ORDER)))
    .sort("status")
    .with_columns(pl.col("status").cast(pl.String))
)
status_counts

status,2025-26,2024-25
str,u32,u32
"""matched""",698310,682528
"""clock differs""",1399,1266
"""under another number""",32,18
"""under another number, other clock""",4,11
"""under another number, other player""",4,1
"""player differs""",60,31
"""kind differs""",6,1
"""not found""",23,21
"""instant replay""",3621,3274


### Where the clock differs

In [4]:
# Where the clock differs: which actions, and by how much (archive minus feed,
# seconds left in the period, so a positive gap is the feed logging it later).
clock_gaps = by_season(
    lambda d: d["rows"]
    .filter(pl.col("status") == "clock differs")
    .group_by("kind")
    .agg(
        pl.len().alias("rows"),
        pl.col("game_id").n_unique().alias("games"),
        pl.col("clock_gap_s").min().round(1).alias("min gap"),
        pl.col("clock_gap_s").median().round(1).alias("median gap"),
        pl.col("clock_gap_s").max().round(1).alias("max gap"),
    )
).sort("season", "rows", descending=[True, True])
clock_gaps

kind,rows,games,min gap,median gap,max gap,season
str,u32,u32,f64,f64,f64,str
"""jumpball""",1357,1297,1.0,4.0,11.0,"""2025-26"""
"""freethrow""",11,5,0.5,1.0,3.0,"""2025-26"""
"""foul""",7,7,0.5,3.0,36.0,"""2025-26"""
"""shot""",6,4,-1.0,6.5,13.0,"""2025-26"""
"""rebound""",6,6,-7.0,1.6,12.0,"""2025-26"""
"""turnover""",4,3,3.0,8.0,15.0,"""2025-26"""
"""steal""",3,2,8.0,8.0,15.0,"""2025-26"""
"""substitution""",2,2,3.0,48.0,93.0,"""2025-26"""
"""violation""",1,1,3.0,3.0,3.0,"""2025-26"""


### What stays open

In [5]:
# What is left open after all three passes, by kind.
OPEN = ["player differs", "kind differs", "not found"]
open_kinds = by_season(
    lambda d: d["rows"]
    .filter(pl.col("status").is_in(OPEN))
    .group_by("status", "kind", "feed_kind")
    .agg(pl.len().alias("rows"), pl.col("game_id").n_unique().alias("games"))
).sort("season", "status", "rows", descending=[True, False, True])
open_kinds

status,kind,feed_kind,rows,games,season
str,str,str,u32,u32,str
"""kind differs""","""shot""","""heave""",6,6,"""2025-26"""
"""not found""","""timeout""",null,6,4,"""2025-26"""
"""not found""","""turnover""",null,5,5,"""2025-26"""
"""not found""","""shot""",null,4,4,"""2025-26"""
"""not found""","""rebound""",null,3,3,"""2025-26"""
"""not found""","""ejection""",null,2,2,"""2025-26"""
"""not found""","""foul""",null,1,1,"""2025-26"""
"""not found""","""steal""",null,1,1,"""2025-26"""
"""not found""","""violation""",null,1,1,"""2025-26"""


### Who took a heave

In [6]:
# The one thing the archive names that the feed does not: who took a heave.
# The feed logs a heave as the team's attempt, with no player on the row.
TEAM_ID_FLOOR = 1610612700
heaves = by_season(
    lambda d: d["rows"]
    .filter((pl.col("kind") == "heave") & pl.col("status").is_in(SETTLED))
    .select(
        pl.len().alias("heaves in both"),
        (pl.col("actor") < TEAM_ID_FLOOR).sum().alias("archive names the player"),
        (pl.col("feed_actor") < TEAM_ID_FLOOR).sum().alias("feed names the player"),
    )
).select("season", pl.exclude("season"))
heaves

season,heaves in both,archive names the player,feed names the player
str,u32,u32,u32
"""2025-26""",1198,1198,0
"""2024-25""",0,0,0


## 3. Feed rows the archive lacks

Feed rows no archive row points at. The feed is the longer of the two by design:
it logs each substitution as two rows, lineup changes between periods, a block or
steal as a row of its own, each player of a double foul, and the end of the game.

In [7]:
def feed_only_rows(d: dict) -> pl.DataFrame:
    """Feed rows no archive row points at, each with the kind of extra it is."""
    rows = d["rows"].filter(pl.col("feed_action_number").is_not_null())
    pointed = rows.filter(
        pl.col("status").is_in([*SETTLED, *PASSES, "player differs"])
    ).select("game_id", pl.col("feed_action_number").alias("action_number"))
    opening = (
        pl.when(pl.col("period") <= REGULATION_PERIODS)
        .then(PERIOD_SECONDS)
        .otherwise(OVERTIME_SECONDS)
    )
    is_sub = pl.col("action_type") == "substitution"
    return d["feed"].join(pointed, on=KEY, how="anti").with_columns(
        pl.when(is_sub & (pl.col("clock_s") == opening))
        .then(pl.lit("lineup change at the start of a period"))
        .when(pl.col("kind") == "substitution in")
        .then(pl.lit("the 'in' row of a substitution"))
        .when((pl.col("kind") == "foul") & (pl.col("descriptor") == "double"))
        .then(pl.lit("the second player of a double foul"))
        .when(pl.col("kind") == "block")
        .then(pl.lit("block, on its own number"))
        .when(pl.col("kind") == "steal")
        .then(pl.lit("steal, on its own number"))
        .when((pl.col("kind") == "game") & (pl.col("sub_type") == "end"))
        .then(pl.lit("end of game"))
        .otherwise(pl.lit("unexpected"))
        .alias("extra")
    )


for d in DATA.values():
    d["feed_only"] = feed_only_rows(d)

feed_only_counts = across_seasons(
    by_season(lambda d: d["feed_only"].group_by("extra").len()), "extra"
).sort(pl.col("extra") == "unexpected", "extra")
feed_only_counts

extra,2025-26,2024-25
str,u32,u32
"""block, on its own number""",13531,13640
"""end of game""",1388,1390
"""lineup change at the start of a period""",26774,24258
"""steal, on its own number""",23451,22955
"""the 'in' row of a substitution""",73077,67569
"""the second player of a double foul""",176,141
"""unexpected""",31,15


In [8]:
unexpected = by_season(
    lambda d: d["feed_only"]
    .filter(pl.col("extra") == "unexpected")
    .group_by("kind", "sub_type")
    .agg(pl.len().alias("rows"), pl.col("game_id").n_unique().alias("games"))
).sort("season", "rows", descending=[True, True])
unexpected

kind,sub_type,rows,games,season
str,str,u32,u32,str
"""heave""","""""",6,6,"""2025-26"""
"""rebound""","""offensive""",5,5,"""2025-26"""
"""shot""","""Jump Shot""",4,4,"""2025-26"""
"""rebound""","""defensive""",4,4,"""2025-26"""
"""shot""","""Layup""",4,4,"""2025-26"""
"""substitution""","""out""",2,1,"""2025-26"""
"""turnover""","""lost ball""",2,2,"""2025-26"""
"""turnover""","""""",1,1,"""2025-26"""
"""turnover""","""out-of-bounds""",1,1,"""2025-26"""


## 4. Period markers

The archive's markers carry Eastern wall-clock time to the minute; the feed stamps
its own to a tenth of a second. Offset is archive minus feed.

In [9]:
MARKER_BAND_S = 60  # a marker reads the minute, so a sound pair sits inside a minute


def marker_offsets(d: dict) -> pl.DataFrame:
    """Each period marker: the archive's minute against the feed's timestamp."""
    wall = pl.col("description").str.extract_groups(WALL_CLOCK)
    hour = wall.struct.field("1").cast(pl.Int64) % 12 + pl.when(
        wall.struct.field("3") == "PM"
    ).then(12).otherwise(0)
    archive = d["archive"].filter(pl.col("kind") == "period").select(
        "game_id",
        "period",
        "sub_type",
        (hour * 3600 + wall.struct.field("2").cast(pl.Int64) * 60).alias("archive_s"),
        pl.col("description").alias("archive_marker"),
    )
    local = pl.col("stamped").dt.convert_time_zone(GAME_TZ)
    feed = d["feed"].filter(pl.col("kind") == "period").select(
        "game_id",
        "period",
        "sub_type",
        (
            local.dt.hour().cast(pl.Int64) * 3600
            + local.dt.minute().cast(pl.Int64) * 60
            + local.dt.second().cast(pl.Int64)
            + local.dt.millisecond() / 1000
        ).alias("feed_s"),
        "time_actual",
    )
    half_day = 12 * 3600
    return archive.join(feed, on=["game_id", "period", "sub_type"], how="inner").with_columns(
        # wrapped so a marker either side of midnight still reads as seconds apart
        (
            (pl.col("archive_s") - pl.col("feed_s") + half_day) % (2 * half_day) - half_day
        ).alias("offset_s")
    )


for d in DATA.values():
    d["markers"] = marker_offsets(d)
    d["markers_out"] = d["markers"].filter(pl.col("offset_s").abs() > MARKER_BAND_S)

marker_spread = by_season(
    lambda d: pl.concat(
        [
            spread(
                d["markers"].filter(
                    (pl.col("sub_type") == t) & (pl.col("offset_s").abs() <= MARKER_BAND_S)
                ),
                "offset_s",
            ).with_columns(
                pl.lit(t).alias("marker"),
                pl.lit(d["markers_out"].filter(pl.col("sub_type") == t).height).alias(
                    "more than a minute apart"
                ),
            )
            for t in ("start", "end")
        ]
    )
).select(
    "season", "marker", "n", "min", "p05", "median", "mean", "p95", "max",
    "more than a minute apart",
)
marker_spread

season,marker,n,min,p05,median,mean,p95,max,more than a minute apart
str,str,u32,f64,f64,f64,f64,f64,f64,i32
"""2025-26""","""start""",5621,-9.9,-7.1,19.4,19.5,46.8,50.0,1
"""2025-26""","""end""",5622,-9.9,-7.2,19.9,19.8,46.8,50.0,0
"""2024-25""","""start""",5634,-9.9,-6.8,20.8,20.5,46.8,50.0,0
"""2024-25""","""end""",5633,-9.9,-7.0,20.1,19.9,47.1,50.0,1


In [10]:
# The markers more than a minute apart, in full: one of the two clocks is wrong here.
by_season(
    lambda d: d["markers_out"].select(
        "game_id", "period", "sub_type", "archive_marker", "time_actual",
        pl.col("offset_s").round(0).alias("offset_s"),
    )
).sort("season", "game_id", "period", "sub_type")

game_id,period,sub_type,archive_marker,time_actual,offset_s,season
str,i64,str,str,str,f64,str
"""0022400239""",4,"""end""","""End of 4th Period (8:37 PM EST)""","""2024-11-18T03:48:50.3Z""",-7910.0,"""2024-25"""
"""0022500232""",4,"""start""","""Start of 4th Period (12:51 AM EST)""","""2025-11-17T03:01:26.1Z""",10174.0,"""2025-26"""


## 5. Timestamp order

`time_actual` is when the scorer logged the action, so feed order and time order
can part. How often, and by how much.

In [11]:
BACK_BUCKETS = ["under 5 s", "5 s to 1 min", "1 to 10 min", "over 10 min"]


def inversions(d: dict) -> pl.DataFrame:
    """Feed rows stamped earlier than the row before them, in feed order."""
    back = (pl.col("before") - pl.col("stamped")).dt.total_milliseconds() / 1000
    return (
        d["feed"]
        .with_columns(pl.col("stamped").shift(1).over("game_id").alias("before"))
        .filter(pl.col("stamped") < pl.col("before"))
        .with_columns(back.alias("seconds_back"))
        .with_columns(
            pl.when(pl.col("seconds_back") < 5)
            .then(pl.lit(BACK_BUCKETS[0]))
            .when(pl.col("seconds_back") < 60)
            .then(pl.lit(BACK_BUCKETS[1]))
            .when(pl.col("seconds_back") < 600)
            .then(pl.lit(BACK_BUCKETS[2]))
            .otherwise(pl.lit(BACK_BUCKETS[3]))
            .alias("how far back")
        )
    )


for d in DATA.values():
    d["inversions"] = inversions(d)

order_summary = by_season(
    lambda d: d["inversions"]
    .group_by("how far back")
    .agg(pl.len().alias("rows"), pl.col("game_id").n_unique().alias("games"))
    .with_columns((pl.col("rows") / d["feed"].height * 100).round(3).alias("% of feed rows"))
).with_columns(pl.col("how far back").cast(pl.Enum(BACK_BUCKETS))).sort(
    "season", "how far back", descending=[True, False]
).select("season", pl.exclude("season"))
order_summary

season,how far back,rows,games,% of feed rows
str,enum,u32,u32,f64
"""2025-26""","""under 5 s""",3751,1281,0.468
"""2025-26""","""5 s to 1 min""",682,518,0.085
"""2025-26""","""1 to 10 min""",43,35,0.005
"""2025-26""","""over 10 min""",4,2,0.0
"""2024-25""","""under 5 s""",3540,1247,0.455
"""2024-25""","""5 s to 1 min""",583,451,0.075
"""2024-25""","""1 to 10 min""",32,32,0.004


## 6. Completeness

In [12]:
def blank(col: str) -> pl.Expr:
    """Rows where a string column is null or empty."""
    return (pl.col(col).is_null() | (pl.col(col) == "")).sum()


completeness = by_season(
    lambda d: d["feed"].select(
        pl.len().alias("feed rows"),
        blank("time_actual").alias("no timestamp"),
        pl.col("stamped").is_null().sum().alias("timestamp unparsed"),
        blank("score_home").alias("no home score"),
        blank("score_away").alias("no away score"),
        blank("clock").alias("no clock"),
    )
).select("season", pl.exclude("season"))
completeness

season,feed rows,no timestamp,timestamp unparsed,no home score,no away score,no clock
str,u32,u32,u32,u32,u32,u32
"""2025-26""",801263,0,0,0,0,0
"""2024-25""",777231,0,0,0,0,0


## 7. Against the box score

Where the two archives credit different players, the official box score is the
referee. Each player's line from the banked game log against what the feed's own
fields give: running totals for points, rebounds, assists, turnovers and fouls,
and a count of its block and steal rows.

In [13]:
BOX_STATS = {"pts": "points", "reb": "rebounds", "ast": "assists", "stl": "steals",
             "blk": "blocks", "tov": "turnovers", "pf": "personal fouls"}


def feed_box(feed: pl.DataFrame) -> pl.DataFrame:
    """Each player's line as the feed's own fields give it."""
    who = ["game_id", "player_id"]
    own = feed.filter(pl.col("person_id") != 0).rename({"person_id": "player_id"})
    totals = own.group_by(who).agg(
        pl.col("points_total").max().alias("pts"),
        pl.col("rebound_total").max().alias("reb"),
        pl.col("turnover_total").max().alias("tov"),
        pl.col("foul_personal_total").max().alias("pf"),
        (pl.col("kind") == "steal").sum().cast(pl.Int64).alias("stl"),
        (pl.col("kind") == "block").sum().cast(pl.Int64).alias("blk"),
    )
    assists = (
        feed.filter(pl.col("assist_person_id").is_not_null())
        .group_by("game_id", pl.col("assist_person_id").alias("player_id"))
        .agg(pl.col("assist_total").max().alias("ast"))
    )
    return totals.join(assists, on=who, how="full", coalesce=True)


def against_the_box(d: dict) -> pl.DataFrame:
    """Per stat: how many player lines the feed reproduces exactly."""
    who = ["game_id", "player_id"]
    lines = (
        d["box"].select(*who, *BOX_STATS)
        .join(feed_box(d["feed"]), on=who, how="left", suffix="_feed")
        .fill_null(0)
    )
    return pl.DataFrame(
        [
            {
                "stat": label,
                "player lines": lines.height,
                "feed agrees": int((lines[stat] == lines[f"{stat}_feed"]).sum()),
                "off by one": int(((lines[stat] - lines[f"{stat}_feed"]).abs() == 1).sum()),
                "off by more": int(((lines[stat] - lines[f"{stat}_feed"]).abs() > 1).sum()),
            }
            for stat, label in BOX_STATS.items()
        ]
    ).with_columns(
        (pl.col("feed agrees") / pl.col("player lines") * 100).round(3).alias("% agrees")
    )


box_check = by_season(against_the_box).select("season", pl.exclude("season"))
box_check

season,stat,player lines,feed agrees,off by one,off by more,% agrees
str,str,i64,i64,i64,i64,f64
"""2025-26""","""points""",30561,30550,5,6,99.964
"""2025-26""","""rebounds""",30561,30514,47,0,99.846
"""2025-26""","""assists""",30561,30546,15,0,99.951
"""2025-26""","""steals""",30561,30534,27,0,99.912
"""2025-26""","""blocks""",30561,30546,15,0,99.951
"""2025-26""","""turnovers""",30561,30542,19,0,99.938
"""2025-26""","""personal fouls""",30561,30557,4,0,99.987
"""2024-25""","""points""",30255,30250,3,2,99.983
"""2024-25""","""rebounds""",30255,30233,22,0,99.927


### Whose version is official

Two checks on the rows left open. Where the archives credit different players on
a counted stat, whose count matches the box score. And how many archive rows the
feed lacks sit beside a feed row the archive lacks: one play, scored two ways.

In [14]:
REFEREED = {"rebound": "reb", "steal": "stl", "block": "blk", "turnover": "tov"}
NEAR_S = 15


def box_sides_with(d: dict) -> pl.DataFrame:
    """Each disputed play of a counted stat: whose count the box score matches.

    A source is right on a play when its count of the stat equals the box
    score's for every player the two sources name. A team has no box line,
    so a play with a team on one side is refereed on the other, and a play
    with a team on both cannot be.
    """
    who = ["game_id", "player_id"]
    disputed = d["rows"].filter(
        (pl.col("status") == "player differs") & pl.col("kind").is_in(list(REFEREED))
    ).select(
        "game_id",
        "action_id",
        "kind",
        pl.col("kind").replace_strict(REFEREED).alias("stat"),
        pl.col("actor").alias("archive_player"),
        # on a block or steal the feed's player is the defender it names
        pl.when(pl.col("kind") == "block")
        .then(pl.col("block_person_id"))
        .when(pl.col("kind") == "steal")
        .then(pl.col("steal_person_id"))
        .otherwise(pl.col("feed_actor"))
        .alias("feed_player"),
    )
    stats = list(REFEREED.values())
    box = d["box"].select(*who, *stats).unpivot(index=who, variable_name="stat", value_name="box")
    feed = (
        feed_box(d["feed"]).select(*who, *stats)
        .unpivot(index=who, variable_name="stat", value_name="feed")
    )
    archive = (
        d["archive"].filter(pl.col("kind").is_in(list(REFEREED)))
        .group_by("game_id", pl.col("actor").alias("player_id"), "kind")
        .agg(pl.len().cast(pl.Int64).alias("archive"))
        .select(*who, pl.col("kind").replace_strict(REFEREED).alias("stat"), "archive")
    )
    counts = (
        box.join(feed, on=[*who, "stat"], how="left")
        .join(archive, on=[*who, "stat"], how="left")
        .fill_null(0)
    )
    named = disputed.unpivot(
        index=["game_id", "action_id", "kind", "stat"],
        on=["archive_player", "feed_player"],
        value_name="player_id",
    ).join(counts, on=[*who, "stat"], how="left")
    return (
        named.group_by("game_id", "action_id", "kind")
        .agg(
            pl.col("box").is_not_null().any().alias("has_box_line"),
            (pl.col("box") == pl.col("feed")).all().alias("feed_right"),
            (pl.col("box") == pl.col("archive")).all().alias("archive_right"),
        )
        .with_columns(
            pl.when(~pl.col("has_box_line"))
            .then(pl.lit("cannot be refereed"))
            .when(pl.col("archive_right") & ~pl.col("feed_right"))
            .then(pl.lit("the archive"))
            .when(pl.col("feed_right") & ~pl.col("archive_right"))
            .then(pl.lit("the feed"))
            .otherwise(pl.lit("neither alone"))
            .alias("box score sides with")
        )
    )


def rescored(d: dict) -> pl.DataFrame:
    """Open archive rows against the unexpected feed rows: the same play scored two ways."""
    open_rows = d["rows"].filter(pl.col("status").is_in(["not found", "kind differs"]))
    extra = d["feed_only"].filter(pl.col("extra") == "unexpected")
    near = (
        open_rows.join(
            extra.select("game_id", "period", pl.col("clock_s").alias("extra_clock_s")),
            on=["game_id", "period"],
            how="inner",
        )
        .filter((pl.col("clock_s") - pl.col("extra_clock_s")).abs() <= NEAR_S)
        .select("game_id", "action_id")
        .unique()
    )
    return pl.DataFrame(
        {
            "archive rows left open": [open_rows.height],
            f"with an unexpected feed row within {NEAR_S} s": [near.height],
            "unexpected feed rows": [extra.height],
        }
    )


SIDES = ["the archive", "the feed", "neither alone", "cannot be refereed"]
referee = (
    by_season(lambda d: box_sides_with(d).group_by("box score sides with").len())
    .pivot(on="box score sides with", index="season", values="len")
    .fill_null(0)
)
referee = referee.with_columns(
    [pl.lit(0, dtype=pl.UInt32).alias(side) for side in SIDES if side not in referee.columns]
).select("season", *SIDES)
rescored_plays = by_season(rescored).select("season", pl.exclude("season"))
print(referee)
print(rescored_plays)

┌─────────┬─────────────┬──────────┬───────────────┬────────────────────┐
│ season  ┆ the archive ┆ the feed ┆ neither alone ┆ cannot be refereed │
│ ---     ┆ ---         ┆ ---      ┆ ---           ┆ ---                │
│ str     ┆ u32         ┆ u32      ┆ u32           ┆ u32                │
╞═════════╪═════════════╪══════════╪═══════════════╪════════════════════╡
│ 2025-26 ┆ 44          ┆ 0        ┆ 0             ┆ 0                  │
│ 2024-25 ┆ 18          ┆ 0        ┆ 0             ┆ 1                  │
└─────────┴─────────────┴──────────┴───────────────┴────────────────────┘
┌─────────┬────────────────────────┬─────────────────────────────────────────┬──────────────────────┐
│ season  ┆ archive rows left open ┆ with an unexpected feed row within 15 s ┆ unexpected feed rows │
│ ---     ┆ ---                    ┆ ---                                     ┆ ---                  │
│ str     ┆ i64                    ┆ i64                                     ┆ i64                  │


## 8. Spot check

2025-26 WCF G7: Castle's tip dunk at Q1 6:33, the timeout after it, and
Holmgren's `in` row at Q4 12:00.

In [15]:
g7 = DATA["2025-26"]["feed"].filter(pl.col("game_id") == SPOT_CHECK_GAME)
eastern = pl.col("stamped").dt.convert_time_zone(GAME_TZ).dt.strftime("%-I:%M:%S %p").alias("eastern")
at_633 = (pl.col("period") == 1) & (pl.col("clock") == "PT06M33.00S")
spot = pl.concat(
    [
        g7.filter(at_633 & pl.col("kind").is_in(["shot", "timeout"])),
        g7.filter(
            (pl.col("period") == 4) & (pl.col("clock") == "PT12M00.00S")
            & (pl.col("sub_type") == "in") & pl.col("player_name_i").str.contains("Holmgren")
        ),
    ]
).select("period", "clock", eastern, "action_type", "sub_type", "qualifiers", "description")
spot

period,clock,eastern,action_type,sub_type,qualifiers,description
i64,str,str,str,str,list[str],str
1,"""PT06M33.00S""","""8:23:08 PM""","""2pt""","""DUNK""","[""2ndchance"", ""fastbreak"", … ""fromturnover""]","""S. Castle tip DUNK (6 PTS)"""
1,"""PT06M33.00S""","""8:23:11 PM""","""timeout""","""full""","[""team"", ""mandatory""]","""OKC Timeout"""
4,"""PT12M00.00S""","""10:12:58 PM""","""substitution""","""in""","[""startperiod""]","""SUB in: C. Holmgren"""


## Reading

Written from the tables above; it opens each season's report.

In [16]:
READING = {
    "2025-26": """
The feed can replace the archive. All 1,387 listed games are banked, each of the
801,263 feed rows carries a timestamp, a score and a clock, and 99.78% of the
archive's rows find the same action on the same number, player, period and clock.
Against the box score the feed reproduces 99.85% to 99.99% of 30,561 player lines,
by stat.

**Where the two differ, largest first**

- **The opening tip, 1,357 rows.** The archive holds a jump ball at the period's
  opening clock. The feed clocks it when the ball is tipped, a median 4 s later.
- **Credit, 60 plays.** The two name different players, mostly on rebounds (23)
  and steals (11). 44 of the 60 are rebounds, steals, blocks or turnovers, which
  the box score counts, and on all 44 the box score sides with the archive. The
  feed is the game as it was scored on the night; corrections made afterwards
  reached the archive and not the feed.
- **Clock, 42 more rows,** by a few seconds.
- **Number, 40 rows** that sit under another action number in the feed.
- **Rows on one side only:** 23 archive rows are not in the feed and 31 feed rows
  are not in the archive. 12 of the 23 sit within 15 s of one of the 31, the same
  play scored two ways. The rest are rows added after the game: timeouts,
  ejections, a technical foul.
- **Six shots at the buzzer** the archive calls a missed shot are a heave in the feed.

**What the feed does not carry**

- Instant replays: 3,621 rows in the archive, none in the feed.
- The player on a heave: all 1,198 name the team only.

**What a reader of the timestamps has to handle**

- 0.56% of rows are stamped earlier than the row before them, 47 by more than a minute.
- Game 0022500232 has timestamps hours off, one by 21 hours, and its fourth-quarter
  start is 2.8 hours from the archive's marker.
- The archive's markers read about 20 s later than the feed's timestamps, inside
  a band of -10 to +50 s.
""",
    "2024-25": """
The feed can replace the archive. All 1,390 listed games are banked, each of the
777,231 feed rows carries a timestamp, a score and a clock, and 99.80% of the
archive's rows find the same action on the same number, player, period and clock.
Against the box score the feed reproduces 99.93% to 100% of 30,255 player lines,
by stat.

**Where the two differ, largest first**

- **The opening tip, 1,248 rows.** The archive holds a jump ball at the period's
  opening clock. The feed clocks it when the ball is tipped, a median 3 s later.
- **Credit, 31 plays.** The two name different players, mostly on rebounds (12)
  and shots (8). 19 of the 31 are rebounds, steals, blocks or turnovers, which the
  box score counts. It sides with the archive on 18; the other names a team on
  both sides and cannot be refereed. The feed is the game as it was scored on the
  night; corrections made afterwards reached the archive and not the feed.
- **Number, 30 rows** that sit under another action number in the feed.
- **Clock, 18 more rows,** by a few seconds.
- **Rows on one side only:** 22 archive rows are not in the feed and 15 feed rows
  are not in the archive. Most of the 22 are rows added after the game: a block
  and its rebound, substitutions, a double technical.

**What the feed does not carry**

- Instant replays: 3,274 rows in the archive, none in the feed.
- Heaves were not logged as their own kind this season, in either source.

**What a reader of the timestamps has to handle**

- 0.53% of rows are stamped earlier than the row before them, 32 by more than a
  minute and none by more than ten.
- Game 0022400239 ends its fourth quarter 2.2 hours from the archive's marker.
- The archive's markers read about 20 s later than the feed's timestamps, inside
  a band of -10 to +50 s.
""",
}

## Write the reports

In [17]:
EXCEPTION_COLUMNS = [
    "side", "status", "game_id", "action_number", "period", "clock", "kind", "actor",
    "description", "feed_action_number", "feed_clock", "feed_kind", "feed_actor",
    "feed_description", "clock_gap_s",
]


def exceptions(d: dict) -> pl.DataFrame:
    """Every archive row that is not plainly matched, and every unexpected feed row."""
    archive_side = d["rows"].filter(
        ~pl.col("status").is_in(["matched", "instant replay"])
    ).with_columns(pl.lit("archive").alias("side"))
    feed_side_rows = (
        d["feed_only"]
        .filter(pl.col("extra") == "unexpected")
        .select(
            pl.lit("feed").alias("side"),
            pl.lit("unexpected").alias("status"),
            "game_id",
            pl.col("action_number").alias("feed_action_number"),
            "period",
            pl.col("clock").alias("feed_clock"),
            pl.col("kind").alias("feed_kind"),
            pl.col("actor").alias("feed_actor"),
            pl.col("description").alias("feed_description"),
        )
    )
    return (
        pl.concat([archive_side, feed_side_rows], how="diagonal_relaxed")
        .select(EXCEPTION_COLUMNS)
        .sort("game_id", "period", "side", "action_number", "feed_action_number")
    )


def md_table(frame: pl.DataFrame) -> str:
    """A frame as a Markdown table, numbers with thousands separators."""
    def cell(value) -> str:
        if value is None:
            return ""
        if isinstance(value, bool):
            return "yes" if value else "no"
        if isinstance(value, int):
            return f"{value:,}"
        if isinstance(value, float):
            return f"{value:,.3f}".rstrip("0").rstrip(".")
        return str(value).replace("|", "/")

    numeric = [dtype.is_numeric() for dtype in frame.dtypes]
    lines = [
        "| " + " | ".join(frame.columns) + " |",
        "|" + "|".join("---:" if n else "---" for n in numeric) + "|",
    ]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in frame.iter_rows()]
    return "\n".join(lines)


def season_of(frame: pl.DataFrame, season: str) -> pl.DataFrame:
    """One season's rows of a by-season table, season column dropped."""
    if "season" in frame.columns:
        return frame.filter(pl.col("season") == season).drop("season")
    return frame.select(frame.columns[0], season)


def report(season: str, reading: str) -> str:
    """The season's report as Markdown."""
    d = DATA[season]
    settled = d["rows"].filter(pl.col("status") != "instant replay").height
    matched = d["rows"].filter(pl.col("status") == "matched").height
    parts = [
        f"# Live feed against the PlayByPlayV3 archive, {season}",
        f"Generated {date.today().isoformat()} by "
        "`notebooks/2025-26/09_live_feed_reconciliation.ipynb`. "
        "Reference only, never published.",
        "## Reading",
        reading.strip(),
        "## 1. Coverage",
        md_table(season_of(coverage, season)),
        "## 2. Archive rows against the feed",
        f"{matched:,} of {settled:,} archive rows ({matched / max(settled, 1):.2%}) find the same "
        "action on the same number, period and clock. Instant replays are set aside: "
        "the feed does not carry them.",
        md_table(season_of(status_counts, season).rename({season: "archive rows"})),
        "### Where the clock differs",
        "Gap is the archive's clock minus the feed's, in seconds left in the period.",
        md_table(season_of(clock_gaps, season)),
        "### What stays open",
        md_table(season_of(open_kinds, season)),
        "### Who took a heave",
        md_table(season_of(heaves, season)),
        "## 3. Feed rows the archive lacks",
        md_table(season_of(feed_only_counts, season).rename({season: "feed rows"})),
        "### Unexpected, by kind",
        md_table(season_of(unexpected, season)),
        "## 4. Period markers",
        "Archive marker (Eastern, to the minute) minus the feed's timestamp, in seconds, "
        "over the markers within a minute of each other.",
        md_table(season_of(marker_spread, season)),
        "## 5. Timestamp order",
        "Feed rows stamped earlier than the row before them, in feed order.",
        md_table(season_of(order_summary, season)),
        "## 6. Completeness",
        md_table(season_of(completeness, season)),
        "## 7. Against the box score",
        "Player lines from the banked game log, against the feed's own running totals "
        "and its block and steal rows.",
        md_table(season_of(box_check, season)),
        "### Where the two credit different players",
        "Disputed rebounds, steals, blocks and turnovers, by whose count of the stat "
        "matches the box score for both players named.",
        md_table(season_of(referee, season)),
        "### The same play, scored two ways",
        md_table(season_of(rescored_plays, season)),
    ]
    if season == "2025-26":
        parts += [
            f"## 8. Spot check, {SPOT_CHECK_GAME}",
            md_table(spot.with_columns(pl.col("qualifiers").list.join(", "))),
        ]
    return "\n\n".join(parts) + "\n"


for season, d in DATA.items():
    out_dir = reference_dir(season) / REPORT_SUBDIR
    out_dir.mkdir(parents=True, exist_ok=True)
    d["exceptions"] = exceptions(d)
    d["exceptions"].write_csv(out_dir / "exceptions.csv")
    (out_dir / "report.md").write_text(report(season, READING[season]))
    print(f"{season}: {out_dir / 'report.md'}")
    print(f"{season}: {out_dir / 'exceptions.csv'} ({d['exceptions'].height:,} rows)")

2025-26: data/2025-26/reference/live_feed_reconciliation/report.md
2025-26: data/2025-26/reference/live_feed_reconciliation/exceptions.csv (1,559 rows)
2024-25: data/2024-25/reference/live_feed_reconciliation/report.md
2024-25: data/2024-25/reference/live_feed_reconciliation/exceptions.csv (1,364 rows)
